# Database and Business Model

This notebook sets up the database used by the AI Analytics Assistant.

The goal is to build a small but realistic e-commerce database with customers, products, orders, payments, returns and stores. I will also check the relationships between the tables and define a few business metrics that the assistant will use later.

Before connecting any AI model, I want to make sure the database itself is clear, reproducible and working correctly.

## 1. Check the project environment

Before starting the database work, I want to check that the project is using the correct Python environment and that the tools I need are available.

Python will be used for generating and validating the data, Git will track the project as it develops, and PostgreSQL will store the business data.

In [1]:
import sys
import shutil
import subprocess

# Show the Python version used by this notebook
print("Python version:", sys.version.split()[0])

# Show the exact Python environment being used
print("Python executable:", sys.executable)

# Check Git
git_version = subprocess.run(
    ["git", "--version"],
    capture_output=True,
    text=True
).stdout.strip()

print("Git:", git_version)

# Check whether PostgreSQL is available
if shutil.which("psql"):
    postgres_version = subprocess.run(
        ["psql", "--version"],
        capture_output=True,
        text=True
    ).stdout.strip()

    print("PostgreSQL:", postgres_version)
else:
    print("PostgreSQL: not found")

Python version: 3.12.11
Python executable: /Users/arvindshine/ai-analytics-assistant/.venv/bin/python
Git: git version 2.50.1 (Apple Git-155)
PostgreSQL: psql (PostgreSQL) 18.6 (Homebrew)


## 2. Start PostgreSQL and test the connection

PostgreSQL is installed, but the database server also needs to be running before I can create or query databases.

In this section, I will start the PostgreSQL service and check that I can connect to it successfully.

In [2]:
# Check that PostgreSQL is running and that we can connect to it
connection_check = subprocess.run(
    [
        "psql",
        "-d", "postgres",
        "-c", "SELECT current_database(), current_user;"
    ],
    capture_output=True,
    text=True
)

print(connection_check.stdout)

# Show any error if the connection failed
if connection_check.returncode != 0:
    print("Connection error:")
    print(connection_check.stderr)

 current_database | current_user 
------------------+--------------
 postgres         | arvindshine
(1 row)




## 3. Plan the database tables

Before creating the tables in PostgreSQL, I want to decide what information each table will store and how the tables will connect to each other.

The database will represent a small e-commerce business with customers, products, orders, payments, returns, stores and promotions. Splitting the data into related tables will make the database easier to manage and will give the assistant realistic SQL problems involving joins and aggregations.

In [3]:
# A simple overview of the tables we plan to create
table_plan = {
    "customers": {
        "stores": "Customer details such as name, region and signup date",
        "primary_key": "customer_id",
        "links_to": []
    },
    "stores": {
        "stores": "Store name, city and region",
        "primary_key": "store_id",
        "links_to": []
    },
    "categories": {
        "stores": "Product categories such as Electronics or Home",
        "primary_key": "category_id",
        "links_to": []
    },
    "products": {
        "stores": "Products, prices and category information",
        "primary_key": "product_id",
        "links_to": ["categories"]
    },
    "orders": {
        "stores": "Customer orders, dates, store and order status",
        "primary_key": "order_id",
        "links_to": ["customers", "stores"]
    },
    "order_items": {
        "stores": "Individual products included in each order",
        "primary_key": "order_item_id",
        "links_to": ["orders", "products", "promotions"]
    },
    "payments": {
        "stores": "Payments made for orders",
        "primary_key": "payment_id",
        "links_to": ["orders"]
    },
    "returns": {
        "stores": "Returned order items and refund information",
        "primary_key": "return_id",
        "links_to": ["order_items"]
    },
    "promotions": {
        "stores": "Discounts and promotional campaigns",
        "primary_key": "promotion_id",
        "links_to": []
    }
}

# Print the planned structure in a readable way
for table_name, details in table_plan.items():
    print(f"{table_name}")
    print(f"  Purpose: {details['stores']}")
    print(f"  Primary key: {details['primary_key']}")
    print(f"  Links to: {', '.join(details['links_to']) if details['links_to'] else 'None'}")
    print()

customers
  Purpose: Customer details such as name, region and signup date
  Primary key: customer_id
  Links to: None

stores
  Purpose: Store name, city and region
  Primary key: store_id
  Links to: None

categories
  Purpose: Product categories such as Electronics or Home
  Primary key: category_id
  Links to: None

products
  Purpose: Products, prices and category information
  Primary key: product_id
  Links to: categories

orders
  Purpose: Customer orders, dates, store and order status
  Primary key: order_id
  Links to: customers, stores

order_items
  Purpose: Individual products included in each order
  Primary key: order_item_id
  Links to: orders, products, promotions

payments
  Purpose: Payments made for orders
  Primary key: payment_id
  Links to: orders

returns
  Purpose: Returned order items and refund information
  Primary key: return_id
  Links to: order_items

promotions
  Purpose: Discounts and promotional campaigns
  Primary key: promotion_id
  Links to: None



## 4. Create the project database

Now that I have a basic plan for the tables, I can create a separate PostgreSQL database for the project.

This database will hold all of the e-commerce tables and data used by the AI Analytics Assistant.

In [4]:
# Name of the PostgreSQL database used by this project
database_name = "ai_analytics"

# Check whether the database already exists
check_database = subprocess.run(
    [
        "psql",
        "-d", "postgres",
        "-tAc",
        f"SELECT 1 FROM pg_database WHERE datname = '{database_name}';"
    ],
    capture_output=True,
    text=True
)

database_exists = check_database.stdout.strip() == "1"

# Create it only if it does not already exist
if database_exists:
    print(f"Database '{database_name}' already exists.")
else:
    create_database = subprocess.run(
        ["createdb", database_name],
        capture_output=True,
        text=True
    )

    if create_database.returncode == 0:
        print(f"Database '{database_name}' created successfully.")
    else:
        print("Database creation failed:")
        print(create_database.stderr)

Database 'ai_analytics' created successfully.


## 5. Create the database tables

Now I can create the tables that will hold the e-commerce data.

Each table will store one type of information, and foreign keys will connect related records. I will also add basic constraints so invalid values cannot easily be inserted into the database.

In [5]:
from pathlib import Path

# SQL that defines the structure of our database
schema_sql = """
CREATE TABLE IF NOT EXISTS customers (
    customer_id INTEGER PRIMARY KEY,
    customer_name VARCHAR(100) NOT NULL,
    city VARCHAR(100) NOT NULL,
    region VARCHAR(50) NOT NULL,
    signup_date DATE NOT NULL
);

CREATE TABLE IF NOT EXISTS stores (
    store_id INTEGER PRIMARY KEY,
    store_name VARCHAR(100) NOT NULL,
    city VARCHAR(100) NOT NULL,
    region VARCHAR(50) NOT NULL,
    opened_date DATE NOT NULL
);

CREATE TABLE IF NOT EXISTS categories (
    category_id INTEGER PRIMARY KEY,
    category_name VARCHAR(100) UNIQUE NOT NULL
);

CREATE TABLE IF NOT EXISTS promotions (
    promotion_id INTEGER PRIMARY KEY,
    promotion_name VARCHAR(100) NOT NULL,
    discount_type VARCHAR(20) NOT NULL,
    discount_value NUMERIC(10, 2) NOT NULL,
    start_date DATE NOT NULL,
    end_date DATE NOT NULL,

    CHECK (discount_type IN ('percentage', 'fixed')),
    CHECK (
        (discount_type = 'percentage' AND discount_value BETWEEN 0 AND 100)
        OR
        (discount_type = 'fixed' AND discount_value >= 0)
    ),
    CHECK (end_date >= start_date)
);

CREATE TABLE IF NOT EXISTS products (
    product_id INTEGER PRIMARY KEY,
    product_name VARCHAR(150) NOT NULL,
    category_id INTEGER NOT NULL,
    list_price NUMERIC(10, 2) NOT NULL,
    cost_price NUMERIC(10, 2) NOT NULL,
    is_active BOOLEAN NOT NULL DEFAULT TRUE,

    FOREIGN KEY (category_id)
        REFERENCES categories(category_id),

    CHECK (list_price >= 0),
    CHECK (cost_price >= 0)
);

CREATE TABLE IF NOT EXISTS orders (
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER NOT NULL,
    store_id INTEGER NOT NULL,
    order_date DATE NOT NULL,
    order_status VARCHAR(20) NOT NULL,

    FOREIGN KEY (customer_id)
        REFERENCES customers(customer_id),

    FOREIGN KEY (store_id)
        REFERENCES stores(store_id),

    CHECK (order_status IN ('completed', 'cancelled'))
);

CREATE TABLE IF NOT EXISTS order_items (
    order_item_id INTEGER PRIMARY KEY,
    order_id INTEGER NOT NULL,
    product_id INTEGER NOT NULL,
    promotion_id INTEGER,
    quantity INTEGER NOT NULL,
    unit_price NUMERIC(10, 2) NOT NULL,
    discount_amount NUMERIC(10, 2) NOT NULL DEFAULT 0,

    FOREIGN KEY (order_id)
        REFERENCES orders(order_id),

    FOREIGN KEY (product_id)
        REFERENCES products(product_id),

    FOREIGN KEY (promotion_id)
        REFERENCES promotions(promotion_id),

    CHECK (quantity > 0),
    CHECK (unit_price >= 0),
    CHECK (discount_amount >= 0)
);

CREATE TABLE IF NOT EXISTS payments (
    payment_id INTEGER PRIMARY KEY,
    order_id INTEGER NOT NULL,
    payment_date DATE NOT NULL,
    payment_method VARCHAR(30) NOT NULL,
    payment_status VARCHAR(30) NOT NULL,
    amount NUMERIC(12, 2) NOT NULL,

    FOREIGN KEY (order_id)
        REFERENCES orders(order_id),

    CHECK (payment_method IN ('card', 'upi', 'wallet', 'bank_transfer')),
    CHECK (
        payment_status IN (
            'paid',
            'failed',
            'refunded',
            'partially_refunded'
        )
    ),
    CHECK (amount >= 0)
);

CREATE TABLE IF NOT EXISTS returns (
    return_id INTEGER PRIMARY KEY,
    order_item_id INTEGER NOT NULL,
    return_date DATE NOT NULL,
    return_quantity INTEGER NOT NULL,
    refund_amount NUMERIC(10, 2) NOT NULL,
    return_reason VARCHAR(100),

    FOREIGN KEY (order_item_id)
        REFERENCES order_items(order_item_id),

    CHECK (return_quantity > 0),
    CHECK (refund_amount >= 0)
);
"""

# Save the SQL so the database structure can be recreated later
schema_path = Path("../database/schema.sql")
schema_path.write_text(schema_sql)

print(f"Schema saved to: {schema_path}")

Schema saved to: ../database/schema.sql


In [6]:
# Run schema.sql against our PostgreSQL database
create_schema = subprocess.run(
    [
        "psql",
        "-d", "ai_analytics",
        "-f", str(schema_path)
    ],
    capture_output=True,
    text=True
)

print(create_schema.stdout)

if create_schema.returncode != 0:
    print("Schema creation error:")
    print(create_schema.stderr)

CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE



## 6. Check the tables and relationships

Now that the tables have been created, I want to check what PostgreSQL actually contains.

I will list the tables and inspect the foreign-key relationships to make sure the database structure matches the design.

In [7]:
# List the tables inside the project database
tables_check = subprocess.run(
    [
        "psql",
        "-d", "ai_analytics",
        "-c",
        """
        SELECT table_name
        FROM information_schema.tables
        WHERE table_schema = 'public'
        ORDER BY table_name;
        """
    ],
    capture_output=True,
    text=True
)

print("Tables:")
print(tables_check.stdout)


# Show the foreign-key relationships between the tables
relationships_check = subprocess.run(
    [
        "psql",
        "-d", "ai_analytics",
        "-c",
        """
        SELECT
            tc.table_name AS child_table,
            kcu.column_name AS foreign_key,
            ccu.table_name AS parent_table,
            ccu.column_name AS referenced_column
        FROM information_schema.table_constraints AS tc
        JOIN information_schema.key_column_usage AS kcu
            ON tc.constraint_name = kcu.constraint_name
            AND tc.constraint_schema = kcu.constraint_schema
        JOIN information_schema.constraint_column_usage AS ccu
            ON ccu.constraint_name = tc.constraint_name
            AND ccu.constraint_schema = tc.constraint_schema
        WHERE tc.constraint_type = 'FOREIGN KEY'
        ORDER BY tc.table_name, kcu.column_name;
        """
    ],
    capture_output=True,
    text=True
)

print("Relationships:")
print(relationships_check.stdout)

Tables:
 table_name  
-------------
 categories
 customers
 order_items
 orders
 payments
 products
 promotions
 returns
 stores
(9 rows)


Relationships:
 child_table |  foreign_key  | parent_table | referenced_column 
-------------+---------------+--------------+-------------------
 order_items | order_id      | orders       | order_id
 order_items | product_id    | products     | product_id
 order_items | promotion_id  | promotions   | promotion_id
 orders      | customer_id   | customers    | customer_id
 orders      | store_id      | stores       | store_id
 payments    | order_id      | orders       | order_id
 products    | category_id   | categories   | category_id
 returns     | order_item_id | order_items  | order_item_id
(8 rows)




## 7. Set the data rules

Before generating the data, I want to fix the size, date range and a few basic rules for the dataset.

These settings will keep the generated data consistent and reproducible. They will also give the project a fixed reference date for questions such as "last month".

In [8]:
from datetime import date

# Using a fixed seed will let us reproduce the same generated data later
RANDOM_SEED = 42

# Approximate size of the dataset
NUM_CUSTOMERS = 5_000
NUM_STORES = 6
NUM_CATEGORIES = 8
NUM_PRODUCTS = 400
NUM_ORDERS = 25_000
NUM_PROMOTIONS = 15

# Dates covered by the dataset
DATA_START_DATE = date(2025, 1, 1)
DATA_END_DATE = date(2026, 7, 31)

# Used later for relative questions such as "last month"
ANALYSIS_REFERENCE_DATE = date(2026, 8, 1)

# Basic order behaviour
CANCELLED_ORDER_RATE = 0.08
PROMOTION_USAGE_RATE = 0.25
RETURN_ITEM_RATE = 0.07

# Number of different products that can appear in one order
MIN_ITEMS_PER_ORDER = 1
MAX_ITEMS_PER_ORDER = 5

# Quantity of each product within an order
MIN_ITEM_QUANTITY = 1
MAX_ITEM_QUANTITY = 3

# Values allowed by our database schema
ORDER_STATUSES = [
    "completed",
    "cancelled"
]

PAYMENT_METHODS = [
    "card",
    "upi",
    "wallet",
    "bank_transfer"
]

PAYMENT_STATUSES = [
    "paid",
    "failed",
    "refunded",
    "partially_refunded"
]

# Print the main settings so they are easy to check
print("Random seed:", RANDOM_SEED)
print()
print("Dataset size")
print("Customers:", NUM_CUSTOMERS)
print("Stores:", NUM_STORES)
print("Categories:", NUM_CATEGORIES)
print("Products:", NUM_PRODUCTS)
print("Orders:", NUM_ORDERS)
print("Promotions:", NUM_PROMOTIONS)

print()
print("Date range")
print("Start date:", DATA_START_DATE)
print("End date:", DATA_END_DATE)
print("Analysis reference date:", ANALYSIS_REFERENCE_DATE)

print()
print("Generation rules")
print(f"Cancelled order rate: {CANCELLED_ORDER_RATE:.0%}")
print(f"Promotion usage rate: {PROMOTION_USAGE_RATE:.0%}")
print(f"Return item rate: {RETURN_ITEM_RATE:.0%}")
print(f"Items per order: {MIN_ITEMS_PER_ORDER}-{MAX_ITEMS_PER_ORDER}")
print(f"Quantity per item: {MIN_ITEM_QUANTITY}-{MAX_ITEM_QUANTITY}")

Random seed: 42

Dataset size
Customers: 5000
Stores: 6
Categories: 8
Products: 400
Orders: 25000
Promotions: 15

Date range
Start date: 2025-01-01
End date: 2026-07-31
Analysis reference date: 2026-08-01

Generation rules
Cancelled order rate: 8%
Promotion usage rate: 25%
Return item rate: 7%
Items per order: 1-5
Quantity per item: 1-3


## 8. Generate the main business data

I will start by generating the tables that do not depend on orders.

These tables contain the customers, stores, product categories, products and promotions that will be used when the transaction data is created later.

In [11]:
import random
from datetime import date, timedelta

import numpy as np
import pandas as pd
from faker import Faker

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
Faker.seed(RANDOM_SEED)

fake = Faker("en_IN")


category_names = [
    "Electronics",
    "Home",
    "Kitchen",
    "Fashion",
    "Beauty",
    "Sports",
    "Books",
    "Accessories"
]

categories_df = pd.DataFrame({
    "category_id": range(1, NUM_CATEGORIES + 1),
    "category_name": category_names
})


store_locations = [
    ("Bengaluru Central", "Bengaluru", "South"),
    ("Chennai City", "Chennai", "South"),
    ("Mumbai Central", "Mumbai", "West"),
    ("Pune City", "Pune", "West"),
    ("Delhi Central", "Delhi", "North"),
    ("Kolkata City", "Kolkata", "East")
]

stores_df = pd.DataFrame([
    {
        "store_id": store_id,
        "store_name": name,
        "city": city,
        "region": region,
        "opened_date": fake.date_between(
            start_date=date(2018, 1, 1),
            end_date=date(2024, 12, 31)
        )
    }
    for store_id, (name, city, region)
    in enumerate(store_locations, start=1)
])


customer_locations = [
    ("Bengaluru", "South"),
    ("Chennai", "South"),
    ("Hyderabad", "South"),
    ("Mumbai", "West"),
    ("Pune", "West"),
    ("Ahmedabad", "West"),
    ("Delhi", "North"),
    ("Jaipur", "North"),
    ("Kolkata", "East")
]

location_weights = [
    0.18, 0.10, 0.12,
    0.16, 0.09, 0.07,
    0.14, 0.06, 0.08
]

customers = []

for customer_id in range(1, NUM_CUSTOMERS + 1):
    city, region = random.choices(
        customer_locations,
        weights=location_weights,
        k=1
    )[0]

    customers.append({
        "customer_id": customer_id,
        "customer_name": fake.name(),
        "city": city,
        "region": region,
        "signup_date": fake.date_between(
            start_date=DATA_START_DATE,
            end_date=DATA_END_DATE
        )
    })

customers_df = pd.DataFrame(customers)


category_price_ranges = {
    1: (500, 50000),
    2: (300, 15000),
    3: (200, 10000),
    4: (300, 8000),
    5: (150, 5000),
    6: (300, 15000),
    7: (100, 2000),
    8: (150, 7500)
}

products = []

for product_id in range(1, NUM_PRODUCTS + 1):
    category_id = random.randint(1, NUM_CATEGORIES)

    min_price, max_price = category_price_ranges[category_id]

    list_price = round(
        random.uniform(min_price, max_price),
        2
    )

    cost_price = round(
        list_price * random.uniform(0.45, 0.75),
        2
    )

    products.append({
        "product_id": product_id,
        "product_name": f"Product {product_id:03d}",
        "category_id": category_id,
        "list_price": list_price,
        "cost_price": cost_price,
        "is_active": random.random() > 0.05
    })

products_df = pd.DataFrame(products)


promotions = []

for promotion_id in range(1, NUM_PROMOTIONS + 1):
    start_date = fake.date_between(
        start_date=DATA_START_DATE,
        end_date=date(2026, 6, 30)
    )

    duration_days = random.randint(7, 30)

    end_date = min(
        start_date + timedelta(days=duration_days),
        DATA_END_DATE
    )

    discount_type = random.choice(["percentage", "fixed"])

    if discount_type == "percentage":
        discount_value = random.choice([5, 10, 15, 20, 25])
    else:
        discount_value = random.choice([100, 250, 500, 750])

    promotions.append({
        "promotion_id": promotion_id,
        "promotion_name": f"Promotion {promotion_id:02d}",
        "discount_type": discount_type,
        "discount_value": discount_value,
        "start_date": start_date,
        "end_date": end_date
    })

promotions_df = pd.DataFrame(promotions)


print("Generated rows")
print("Customers:", len(customers_df))
print("Stores:", len(stores_df))
print("Categories:", len(categories_df))
print("Products:", len(products_df))
print("Promotions:", len(promotions_df))

Generated rows
Customers: 5000
Stores: 6
Categories: 8
Products: 400
Promotions: 15


## 9. Check the generated data

Before loading the data into PostgreSQL, I want to check a few rows and make sure the values look sensible.

This gives me a chance to catch obvious problems before the data is added to the database.

In [12]:
# Check the size of each generated table
print("Table sizes")
print("Customers:", customers_df.shape)
print("Stores:", stores_df.shape)
print("Categories:", categories_df.shape)
print("Products:", products_df.shape)
print("Promotions:", promotions_df.shape)

print("\nCustomer sample")
display(customers_df.head())

print("\nProduct sample")
display(products_df.head())

print("\nPromotion sample")
display(promotions_df.head())

print("\nCustomers by region")
print(customers_df["region"].value_counts())

print("\nProduct active status")
print(products_df["is_active"].value_counts())

print("\nBasic checks")

print(
    "Customer signup dates valid:",
    customers_df["signup_date"].between(
        DATA_START_DATE,
        DATA_END_DATE
    ).all()
)

print(
    "Product prices valid:",
    (products_df["list_price"] >= products_df["cost_price"]).all()
)

print(
    "Promotion dates valid:",
    (promotions_df["end_date"] >= promotions_df["start_date"]).all()
)

Table sizes
Customers: (5000, 5)
Stores: (6, 5)
Categories: (8, 2)
Products: (400, 6)
Promotions: (15, 6)

Customer sample


,customer_id,customer_name,city,region,signup_date
0,1,Pahal Balay,Pune,West,2025-01-18
1,2,Viraj Tiwari,Bengaluru,South,2025-12-13
2,3,Oni Kannan,Chennai,South,2025-09-16
3,4,Abeer Dutta,Chennai,South,2026-02-07
4,5,Arunima Dugal,Delhi,North,2025-05-05



Product sample


,product_id,product_name,category_id,list_price,cost_price,is_active
0,1,Product 001,3,6220.04,3725.30,True
1,2,Product 002,8,3769.91,1899.93,True
2,3,Product 003,5,4899.40,2207.05,False
3,4,Product 004,2,11140.79,5706.30,True
4,5,Product 005,7,886.68,487.80,True



Promotion sample


,promotion_id,promotion_name,discount_type,discount_value,start_date,end_date
0,1,Promotion 01,fixed,500,2025-08-11,2025-08-19
1,2,Promotion 02,fixed,750,2025-01-30,2025-02-11
2,3,Promotion 03,fixed,100,2025-03-11,2025-03-21
3,4,Promotion 04,fixed,750,2025-05-13,2025-06-03
4,5,Promotion 05,fixed,100,2025-03-20,2025-04-10



Customers by region
region
South    1996
West     1627
North     981
East      396
Name: count, dtype: int64

Product active status
is_active
True     388
False     12
Name: count, dtype: int64

Basic checks
Customer signup dates valid: True
Product prices valid: True
Promotion dates valid: True


## 10. Generate the orders

Now I will generate the customer orders.

The order dates must come after each customer's signup date, and most customers should buy from a store in their own region. I will also include some seasonal variation and cancelled orders so the data is not completely uniform.

In [13]:
from bisect import bisect_left

# Separate random generator for orders so this section is reproducible
order_random = random.Random(RANDOM_SEED + 1)
order_rng = np.random.default_rng(RANDOM_SEED + 1)

# Give customers different buying activity levels
customer_order_weights = order_rng.lognormal(
    mean=0.0,
    sigma=0.9,
    size=NUM_CUSTOMERS
)

customer_probabilities = (
    customer_order_weights / customer_order_weights.sum()
)

# Choose the customer for each order
selected_customer_ids = order_rng.choice(
    np.arange(1, NUM_CUSTOMERS + 1),
    size=NUM_ORDERS,
    p=customer_probabilities
)

# Create all possible order dates
all_order_dates = [
    timestamp.date()
    for timestamp in pd.date_range(
        DATA_START_DATE,
        DATA_END_DATE,
        freq="D"
    )
]

# Give some periods slightly more sales activity
date_weights = []

for order_date in all_order_dates:
    weight = 1.0

    # Higher activity near the end of the year
    if order_date.month in [10, 11, 12]:
        weight *= 1.35

    # Slightly more activity on weekends
    if order_date.weekday() >= 5:
        weight *= 1.10

    date_weights.append(weight)


customer_signup = (
    customers_df
    .set_index("customer_id")["signup_date"]
    .to_dict()
)

customer_region = (
    customers_df
    .set_index("customer_id")["region"]
    .to_dict()
)

stores_by_region = (
    stores_df
    .groupby("region")["store_id"]
    .apply(list)
    .to_dict()
)

all_store_ids = stores_df["store_id"].tolist()

orders = []

for order_id, customer_id in enumerate(
    selected_customer_ids,
    start=1
):
    customer_id = int(customer_id)

    signup_date = customer_signup[customer_id]

    # Find the first possible order date after signup
    start_index = bisect_left(
        all_order_dates,
        signup_date
    )

    eligible_dates = all_order_dates[start_index:]
    eligible_weights = date_weights[start_index:]

    order_date = order_random.choices(
        eligible_dates,
        weights=eligible_weights,
        k=1
    )[0]

    region = customer_region[customer_id]
    regional_stores = stores_by_region.get(region, [])

    # Most orders are assigned to a store in the customer's region
    if regional_stores and order_random.random() < 0.85:
        store_id = order_random.choice(regional_stores)
    else:
        store_id = order_random.choice(all_store_ids)

    if order_random.random() < CANCELLED_ORDER_RATE:
        order_status = "cancelled"
    else:
        order_status = "completed"

    orders.append({
        "order_id": order_id,
        "customer_id": customer_id,
        "store_id": store_id,
        "order_date": order_date,
        "order_status": order_status
    })

orders_df = pd.DataFrame(orders)


print("Orders generated:", len(orders_df))
print("Unique customers with orders:", orders_df["customer_id"].nunique())
print(
    "Customers with no orders:",
    NUM_CUSTOMERS - orders_df["customer_id"].nunique()
)

print()
print("Order status")
print(orders_df["order_status"].value_counts())

print()
cancelled_rate = (
    orders_df["order_status"].eq("cancelled").mean()
)

print(f"Actual cancelled rate: {cancelled_rate:.2%}")

# Check that nobody ordered before signing up
order_date_check = orders_df.merge(
    customers_df[["customer_id", "signup_date"]],
    on="customer_id"
)

valid_order_dates = (
    order_date_check["order_date"]
    >= order_date_check["signup_date"]
).all()

print("All orders occur after signup:", valid_order_dates)

Orders generated: 25000
Unique customers with orders: 4452
Customers with no orders: 548

Order status
order_status
completed    23042
cancelled     1958
Name: count, dtype: int64

Actual cancelled rate: 7.83%
All orders occur after signup: True


## 11. Generate the order items

An order can contain more than one product, so I need a separate table to store the individual items inside each order.

This section will connect orders to products, add quantities and prices, and apply promotions only when they are active on the order date.

In [14]:
items_random = random.Random(RANDOM_SEED + 2)
items_rng = np.random.default_rng(RANDOM_SEED + 2)

active_products = products_df[
    products_df["is_active"] == True
].copy()

active_product_ids = active_products["product_id"].to_numpy()

product_prices = (
    products_df
    .set_index("product_id")["list_price"]
    .to_dict()
)

# Give some products more demand than others
product_weights = items_rng.lognormal(
    mean=0.0,
    sigma=0.8,
    size=len(active_product_ids)
)

product_probabilities = (
    product_weights / product_weights.sum()
)

promotion_records = promotions_df.to_dict("records")

order_items = []
order_item_id = 1

for order in orders_df.itertuples(index=False):

    num_items = items_random.randint(
        MIN_ITEMS_PER_ORDER,
        MAX_ITEMS_PER_ORDER
    )

    # Choose different products within the same order
    selected_products = items_rng.choice(
        active_product_ids,
        size=num_items,
        replace=False,
        p=product_probabilities
    )

    for product_id in selected_products:

        product_id = int(product_id)

        quantity = items_random.randint(
            MIN_ITEM_QUANTITY,
            MAX_ITEM_QUANTITY
        )

        unit_price = round(
            float(product_prices[product_id]),
            2
        )

        promotion_id = None
        discount_amount = 0.0

        # Find promotions that were active when the order was placed
        eligible_promotions = [
            promotion
            for promotion in promotion_records
            if promotion["start_date"] <= order.order_date <= promotion["end_date"]
        ]

        # Apply a promotion to some eligible items
        if (
            eligible_promotions
            and items_random.random() < PROMOTION_USAGE_RATE
        ):
            promotion = items_random.choice(
                eligible_promotions
            )

            promotion_id = promotion["promotion_id"]

            line_value = unit_price * quantity

            if promotion["discount_type"] == "percentage":
                discount_amount = round(
                    line_value
                    * float(promotion["discount_value"])
                    / 100,
                    2
                )
            else:
                discount_amount = round(
                    min(
                        float(promotion["discount_value"]),
                        line_value * 0.50
                    ),
                    2
                )

        order_items.append({
            "order_item_id": order_item_id,
            "order_id": order.order_id,
            "product_id": product_id,
            "promotion_id": promotion_id,
            "quantity": quantity,
            "unit_price": unit_price,
            "discount_amount": discount_amount
        })

        order_item_id += 1

order_items_df = pd.DataFrame(order_items)


print("Order items generated:", len(order_items_df))

print(
    "Average different products per order:",
    round(
        len(order_items_df) / len(orders_df),
        2
    )
)

promotion_items = (
    order_items_df["promotion_id"]
    .notna()
    .sum()
)

promotion_rate = (
    order_items_df["promotion_id"]
    .notna()
    .mean()
)

print("Items using promotions:", promotion_items)
print(f"Actual promotion usage rate: {promotion_rate:.2%}")

print(
    "All quantities are positive:",
    (order_items_df["quantity"] > 0).all()
)

print(
    "All discounts are non-negative:",
    (order_items_df["discount_amount"] >= 0).all()
)

line_values = (
    order_items_df["unit_price"]
    * order_items_df["quantity"]
)

print(
    "No discount exceeds the item value:",
    (order_items_df["discount_amount"] <= line_values).all()
)

Order items generated: 75104
Average different products per order: 3.0
Items using promotions: 7462
Actual promotion usage rate: 9.94%
All quantities are positive: True
All discounts are non-negative: True
No discount exceeds the item value: True


## 12. Generate returns

Some completed purchases will be returned, so I will create a smaller returns dataset linked to the original order items.

A return must belong to a completed order, cannot return more units than were purchased, and the refund amount should be based on what the customer actually paid for that item.

In [15]:
returns_random = random.Random(RANDOM_SEED + 3)

# Add the order date and status to each order item
return_candidates = order_items_df.merge(
    orders_df[["order_id", "order_date", "order_status"]],
    on="order_id",
    how="left"
)

# Only completed orders can be returned
# We also need at least one day available for the return date
return_candidates = return_candidates[
    (return_candidates["order_status"] == "completed")
    & (return_candidates["order_date"] < DATA_END_DATE)
].copy()

returns = []
return_id = 1

for item in return_candidates.itertuples(index=False):

    if returns_random.random() < RETURN_ITEM_RATE:

        # A customer can return some or all of the purchased quantity
        return_quantity = returns_random.randint(
            1,
            item.quantity
        )

        # Refund the returned share of the discounted line value
        line_value = item.unit_price * item.quantity
        net_line_value = line_value - item.discount_amount

        refund_amount = round(
            (net_line_value / item.quantity) * return_quantity,
            2
        )

        # Returns happen between 1 and 30 days after the order
        latest_return_date = min(
            item.order_date + timedelta(days=30),
            DATA_END_DATE
        )

        days_available = (
            latest_return_date - item.order_date
        ).days

        return_date = (
            item.order_date
            + timedelta(
                days=returns_random.randint(1, days_available)
            )
        )

        return_reason = returns_random.choice([
            "Damaged item",
            "Wrong item",
            "Changed mind",
            "Size or fit issue",
            "Not as expected"
        ])

        returns.append({
            "return_id": return_id,
            "order_item_id": item.order_item_id,
            "return_date": return_date,
            "return_quantity": return_quantity,
            "refund_amount": refund_amount,
            "return_reason": return_reason
        })

        return_id += 1

returns_df = pd.DataFrame(returns)


print("Returns generated:", len(returns_df))

actual_return_rate = (
    len(returns_df) / len(return_candidates)
)

print(f"Actual eligible item return rate: {actual_return_rate:.2%}")


# Check returned quantities against what was originally purchased
return_check = returns_df.merge(
    order_items_df[
        ["order_item_id", "quantity"]
    ],
    on="order_item_id",
    how="left"
)

print(
    "Returned quantities are valid:",
    (
        return_check["return_quantity"]
        <= return_check["quantity"]
    ).all()
)

print(
    "All refunds are non-negative:",
    (returns_df["refund_amount"] >= 0).all()
)


# Check that returns happen after the original order
return_date_check = (
    returns_df
    .merge(
        order_items_df[
            ["order_item_id", "order_id"]
        ],
        on="order_item_id"
    )
    .merge(
        orders_df[
            ["order_id", "order_date", "order_status"]
        ],
        on="order_id"
    )
)

print(
    "All returns are from completed orders:",
    (
        return_date_check["order_status"]
        == "completed"
    ).all()
)

print(
    "All returns occur after the order date:",
    (
        return_date_check["return_date"]
        > return_date_check["order_date"]
    ).all()
)

Returns generated: 4832
Actual eligible item return rate: 7.05%
Returned quantities are valid: True
All refunds are non-negative: True
All returns are from completed orders: True
All returns occur after the order date: True


## 13. Generate payments

I will now create one payment record for each order.

Completed orders will have a payment based on the value of the items after discounts. If an order has returns, its payment status will show whether it was partially or fully refunded. Cancelled orders will not have a successful payment.

In [16]:
payments_random = random.Random(RANDOM_SEED + 4)

# Calculate the amount paid for each order before refunds
order_items_with_value = order_items_df.copy()

order_items_with_value["net_item_value"] = (
    order_items_with_value["unit_price"]
    * order_items_with_value["quantity"]
    - order_items_with_value["discount_amount"]
)

order_totals = (
    order_items_with_value
    .groupby("order_id")["net_item_value"]
    .sum()
    .to_dict()
)

# Calculate the total refund for each order
refunds_by_order = (
    returns_df
    .merge(
        order_items_df[["order_item_id", "order_id"]],
        on="order_item_id",
        how="left"
    )
    .groupby("order_id")["refund_amount"]
    .sum()
    .to_dict()
)

payment_methods = [
    "card",
    "upi",
    "wallet",
    "bank_transfer"
]

payment_method_weights = [
    0.35,
    0.45,
    0.12,
    0.08
]

payments = []

for payment_id, order in enumerate(
    orders_df.itertuples(index=False),
    start=1
):
    order_total = round(
        float(order_totals[order.order_id]),
        2
    )

    refund_total = round(
        float(refunds_by_order.get(order.order_id, 0)),
        2
    )

    payment_method = payments_random.choices(
        payment_methods,
        weights=payment_method_weights,
        k=1
    )[0]

    if order.order_status == "cancelled":
        payment_status = "failed"
        payment_amount = 0.0

    elif refund_total == 0:
        payment_status = "paid"
        payment_amount = order_total

    elif abs(refund_total - order_total) < 0.01:
        payment_status = "refunded"
        payment_amount = order_total

    else:
        payment_status = "partially_refunded"
        payment_amount = order_total

    payments.append({
        "payment_id": payment_id,
        "order_id": order.order_id,
        "payment_date": order.order_date,
        "payment_method": payment_method,
        "payment_status": payment_status,
        "amount": payment_amount
    })

payments_df = pd.DataFrame(payments)


print("Payments generated:", len(payments_df))

print()
print("Payment status")
print(payments_df["payment_status"].value_counts())

print()
print("Payment methods")
print(payments_df["payment_method"].value_counts())

print()
print(
    "One payment per order:",
    len(payments_df) == len(orders_df)
)

cancelled_payment_check = (
    payments_df
    .merge(
        orders_df[["order_id", "order_status"]],
        on="order_id"
    )
)

print(
    "Cancelled orders have failed payments:",
    (
        cancelled_payment_check.loc[
            cancelled_payment_check["order_status"] == "cancelled",
            "payment_status"
        ] == "failed"
    ).all()
)

print(
    "Cancelled orders have zero payment amount:",
    (
        cancelled_payment_check.loc[
            cancelled_payment_check["order_status"] == "cancelled",
            "amount"
        ] == 0
    ).all()
)

print(
    "All payment amounts are non-negative:",
    (payments_df["amount"] >= 0).all()
)

Payments generated: 25000

Payment status
payment_status
paid                  18609
partially_refunded     4226
failed                 1958
refunded                207
Name: count, dtype: int64

Payment methods
payment_method
upi              11276
card              8771
wallet            2934
bank_transfer     2019
Name: count, dtype: int64

One payment per order: True
Cancelled orders have failed payments: True
Cancelled orders have zero payment amount: True
All payment amounts are non-negative: True


## 14. Load the data into PostgreSQL

The generated data currently exists only as pandas DataFrames in Python.

I will now load each dataset into PostgreSQL in the correct order so the foreign-key relationships are preserved. After loading, I will check the number of rows stored in each table.

In [17]:
import psycopg

# Convert pandas/numpy values into normal Python values for PostgreSQL
def clean_value(value):
    if pd.isna(value):
        return None

    if isinstance(value, np.generic):
        return value.item()

    return value


# Insert a DataFrame into a PostgreSQL table
def insert_dataframe(cursor, table_name, dataframe):
    columns = list(dataframe.columns)

    column_names = ", ".join(columns)
    placeholders = ", ".join(["%s"] * len(columns))

    query = f"""
        INSERT INTO {table_name} ({column_names})
        VALUES ({placeholders})
    """

    rows = [
        tuple(clean_value(value) for value in row)
        for row in dataframe.itertuples(index=False, name=None)
    ]

    cursor.executemany(query, rows)

    print(f"Loaded {len(rows):,} rows into {table_name}")


# Open a connection to the project database
with psycopg.connect(dbname="ai_analytics") as conn:

    with conn.cursor() as cur:

        # Clear existing data so this cell can be rerun safely
        cur.execute("""
            TRUNCATE TABLE
                returns,
                payments,
                order_items,
                orders,
                products,
                promotions,
                categories,
                stores,
                customers
            CASCADE;
        """)

        # Parent tables must be loaded before tables that depend on them
        insert_dataframe(cur, "customers", customers_df)
        insert_dataframe(cur, "stores", stores_df)
        insert_dataframe(cur, "categories", categories_df)
        insert_dataframe(cur, "promotions", promotions_df)
        insert_dataframe(cur, "products", products_df)

        # Transaction tables
        insert_dataframe(cur, "orders", orders_df)
        insert_dataframe(cur, "order_items", order_items_df)
        insert_dataframe(cur, "payments", payments_df)
        insert_dataframe(cur, "returns", returns_df)


print("\nDatabase row counts")

# Connect again and check what PostgreSQL actually contains
with psycopg.connect(dbname="ai_analytics") as conn:

    with conn.cursor() as cur:

        table_names = [
            "customers",
            "stores",
            "categories",
            "promotions",
            "products",
            "orders",
            "order_items",
            "payments",
            "returns"
        ]

        for table_name in table_names:
            cur.execute(
                f"SELECT COUNT(*) FROM {table_name};"
            )

            row_count = cur.fetchone()[0]

            print(f"{table_name}: {row_count:,}")

Loaded 5,000 rows into customers
Loaded 6 rows into stores
Loaded 8 rows into categories
Loaded 15 rows into promotions
Loaded 400 rows into products
Loaded 25,000 rows into orders
Loaded 75,104 rows into order_items
Loaded 25,000 rows into payments
Loaded 4,832 rows into returns

Database row counts
customers: 5,000
stores: 6
categories: 8
promotions: 15
products: 400
orders: 25,000
order_items: 75,104
payments: 25,000
returns: 4,832


## 15. Validate the loaded database

Now that the data is stored in PostgreSQL, I want to check that the loaded tables still follow the rules used during generation.

I will compare the row counts and check for problems such as invalid dates, broken relationships, incorrect returns and cancelled orders with successful payments.

In [18]:
table_frames = {
    "customers": customers_df,
    "stores": stores_df,
    "categories": categories_df,
    "promotions": promotions_df,
    "products": products_df,
    "orders": orders_df,
    "order_items": order_items_df,
    "payments": payments_df,
    "returns": returns_df
}

all_checks_passed = True

with psycopg.connect(dbname="ai_analytics") as conn:
    with conn.cursor() as cur:

        print("Row count checks")

        for table_name, dataframe in table_frames.items():
            cur.execute(f"SELECT COUNT(*) FROM {table_name};")
            database_count = cur.fetchone()[0]

            expected_count = len(dataframe)
            passed = database_count == expected_count

            print(
                f"{table_name}: {database_count:,} "
                f"(expected {expected_count:,}) - {passed}"
            )

            if not passed:
                all_checks_passed = False

        print("\nData integrity checks")

        # Orders should never happen before the customer signed up
        cur.execute("""
            SELECT COUNT(*)
            FROM orders o
            JOIN customers c
                ON o.customer_id = c.customer_id
            WHERE o.order_date < c.signup_date;
        """)

        invalid_order_dates = cur.fetchone()[0]

        print(
            "Orders before customer signup:",
            invalid_order_dates
        )

        if invalid_order_dates != 0:
            all_checks_passed = False

        # Cancelled orders should have failed payments with zero amount
        cur.execute("""
            SELECT COUNT(*)
            FROM orders o
            JOIN payments p
                ON o.order_id = p.order_id
            WHERE o.order_status = 'cancelled'
              AND (
                  p.payment_status <> 'failed'
                  OR p.amount <> 0
              );
        """)

        invalid_cancelled_payments = cur.fetchone()[0]

        print(
            "Invalid cancelled-order payments:",
            invalid_cancelled_payments
        )

        if invalid_cancelled_payments != 0:
            all_checks_passed = False

        # A return cannot contain more units than were purchased
        cur.execute("""
            SELECT COUNT(*)
            FROM returns r
            JOIN order_items oi
                ON r.order_item_id = oi.order_item_id
            WHERE r.return_quantity > oi.quantity;
        """)

        invalid_return_quantities = cur.fetchone()[0]

        print(
            "Returns exceeding purchased quantity:",
            invalid_return_quantities
        )

        if invalid_return_quantities != 0:
            all_checks_passed = False

        # Returns should only belong to completed orders
        cur.execute("""
            SELECT COUNT(*)
            FROM returns r
            JOIN order_items oi
                ON r.order_item_id = oi.order_item_id
            JOIN orders o
                ON oi.order_id = o.order_id
            WHERE o.order_status <> 'completed';
        """)

        invalid_return_orders = cur.fetchone()[0]

        print(
            "Returns from non-completed orders:",
            invalid_return_orders
        )

        if invalid_return_orders != 0:
            all_checks_passed = False

        # Return date must be after the original order date
        cur.execute("""
            SELECT COUNT(*)
            FROM returns r
            JOIN order_items oi
                ON r.order_item_id = oi.order_item_id
            JOIN orders o
                ON oi.order_id = o.order_id
            WHERE r.return_date <= o.order_date;
        """)

        invalid_return_dates = cur.fetchone()[0]

        print(
            "Invalid return dates:",
            invalid_return_dates
        )

        if invalid_return_dates != 0:
            all_checks_passed = False

        # Promotions should only be used while they are active
        cur.execute("""
            SELECT COUNT(*)
            FROM order_items oi
            JOIN orders o
                ON oi.order_id = o.order_id
            JOIN promotions p
                ON oi.promotion_id = p.promotion_id
            WHERE oi.promotion_id IS NOT NULL
              AND (
                  o.order_date < p.start_date
                  OR o.order_date > p.end_date
              );
        """)

        invalid_promotions = cur.fetchone()[0]

        print(
            "Promotions used outside active dates:",
            invalid_promotions
        )

        if invalid_promotions != 0:
            all_checks_passed = False


print("\nOverall database validation:", all_checks_passed)

Row count checks
customers: 5,000 (expected 5,000) - True
stores: 6 (expected 6) - True
categories: 8 (expected 8) - True
promotions: 15 (expected 15) - True
products: 400 (expected 400) - True
orders: 25,000 (expected 25,000) - True
order_items: 75,104 (expected 75,104) - True
payments: 25,000 (expected 25,000) - True
returns: 4,832 (expected 4,832) - True

Data integrity checks
Orders before customer signup: 0
Invalid cancelled-order payments: 0
Returns exceeding purchased quantity: 0
Returns from non-completed orders: 0
Invalid return dates: 0
Promotions used outside active dates: 0

Overall database validation: True


## 16. Try some basic SQL queries

Before using an AI model to generate SQL, I want to understand how the database can be queried myself.

I will start with a few simple business questions using filters, joins, grouping and aggregation.

In [19]:
def run_query(query):
    with psycopg.connect(dbname="ai_analytics") as conn:
        return pd.read_sql_query(query, conn)


total_orders = run_query("""
    SELECT COUNT(*) AS total_orders
    FROM orders;
""")

orders_by_status = run_query("""
    SELECT
        order_status,
        COUNT(*) AS order_count
    FROM orders
    GROUP BY order_status
    ORDER BY order_count DESC;
""")

orders_by_region = run_query("""
    SELECT
        c.region,
        COUNT(DISTINCT o.order_id) AS order_count
    FROM orders o
    JOIN customers c
        ON o.customer_id = c.customer_id
    WHERE o.order_status = 'completed'
    GROUP BY c.region
    ORDER BY order_count DESC;
""")

top_products = run_query("""
    SELECT
        p.product_name,
        SUM(oi.quantity) AS units_sold
    FROM order_items oi
    JOIN orders o
        ON oi.order_id = o.order_id
    JOIN products p
        ON oi.product_id = p.product_id
    WHERE o.order_status = 'completed'
    GROUP BY p.product_id, p.product_name
    ORDER BY units_sold DESC
    LIMIT 10;
""")

print("Total orders")
display(total_orders)

print("Orders by status")
display(orders_by_status)

print("Completed orders by customer region")
display(orders_by_region)

print("Top 10 products by units sold")
display(top_products)

Total orders


/var/folders/j9/c4fkmh8s679bzbcy5bdmsq300000gn/T/ipykernel_8751/2183512943.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(query, conn)


,total_orders
0,25000


Orders by status


,order_status,order_count
0,completed,23042
1,cancelled,1958


Completed orders by customer region


,region,order_count
0,South,9269
1,West,7507
2,North,4468
3,East,1798


Top 10 products by units sold


,product_name,units_sold
0,Product 138,3741
1,Product 352,2701
2,Product 183,2689
3,Product 300,1694
4,Product 099,1372
5,Product 038,1346
6,Product 258,1258
7,Product 113,1210
8,Product 287,1181
9,Product 146,1133


## 17. Define the business glossary

Business terms can have different meanings depending on the company.

I will define the main metrics used in this project so the assistant has a consistent meaning for terms such as revenue, order count and repeat customer. These definitions will also help avoid unnecessary ambiguity later.

In [20]:
import json

business_glossary = {
    "gross_sales": {
        "definition": "Total item value before discounts and refunds.",
        "calculation": "SUM(unit_price * quantity)",
        "rules": [
            "Only completed orders are included."
        ]
    },

    "discount_amount": {
        "definition": "Total discounts applied to completed order items.",
        "calculation": "SUM(discount_amount)",
        "rules": [
            "Only completed orders are included."
        ]
    },

    "refund_amount": {
        "definition": "Total amount refunded for returned items.",
        "calculation": "SUM(refund_amount)",
        "rules": [
            "Refunds come from the returns table."
        ]
    },

    "net_revenue": {
        "definition": "Revenue after discounts and refunds.",
        "calculation": "gross_sales - discount_amount - refund_amount",
        "rules": [
            "Cancelled orders are excluded.",
            "Revenue in this project means net revenue unless the user asks for gross sales."
        ]
    },

    "order_count": {
        "definition": "Number of completed customer orders.",
        "calculation": "COUNT(DISTINCT order_id)",
        "rules": [
            "Cancelled orders are excluded."
        ]
    },

    "average_order_value": {
        "definition": "Average net revenue generated per completed order.",
        "calculation": "net_revenue / order_count",
        "rules": [
            "Cancelled orders are excluded."
        ]
    },

    "repeat_customer": {
        "definition": "A customer with at least two completed orders in the selected period.",
        "calculation": "completed order count >= 2",
        "rules": [
            "The selected analysis period must be applied before counting orders."
        ]
    },

    "active_customer": {
        "definition": "A customer with at least one completed order in the selected period.",
        "calculation": "completed order count >= 1",
        "rules": [
            "The selected analysis period must be applied before counting orders."
        ]
    }
}

glossary_path = Path("../config/business_glossary.json")

with open(glossary_path, "w") as file:
    json.dump(business_glossary, file, indent=4)

print("Business glossary saved to:", glossary_path)
print("Metrics defined:", len(business_glossary))

print("\nMetric names")
for metric in business_glossary:
    print("-", metric)

print("\nAnalysis reference date:", ANALYSIS_REFERENCE_DATE)

Business glossary saved to: ../config/business_glossary.json
Metrics defined: 8

Metric names
- gross_sales
- discount_amount
- refund_amount
- net_revenue
- order_count
- average_order_value
- repeat_customer
- active_customer

Analysis reference date: 2026-08-01


## 18. Add useful indexes

The database will often search and join using columns such as order dates, customer IDs and product IDs.

I will add a few indexes to these commonly used columns so PostgreSQL can find related records more efficiently as the assistant starts running analytical queries.

In [21]:
index_sql = """
CREATE INDEX IF NOT EXISTS idx_orders_order_date
ON orders(order_date);

CREATE INDEX IF NOT EXISTS idx_orders_customer_id
ON orders(customer_id);

CREATE INDEX IF NOT EXISTS idx_orders_store_id
ON orders(store_id);

CREATE INDEX IF NOT EXISTS idx_order_items_order_id
ON order_items(order_id);

CREATE INDEX IF NOT EXISTS idx_order_items_product_id
ON order_items(product_id);

CREATE INDEX IF NOT EXISTS idx_products_category_id
ON products(category_id);

CREATE INDEX IF NOT EXISTS idx_returns_order_item_id
ON returns(order_item_id);
"""

with psycopg.connect(dbname="ai_analytics") as conn:
    with conn.cursor() as cur:
        cur.execute(index_sql)

print("Indexes created successfully.")


# Check the indexes that now exist
with psycopg.connect(dbname="ai_analytics") as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT
                tablename,
                indexname
            FROM pg_indexes
            WHERE schemaname = 'public'
            ORDER BY tablename, indexname;
        """)

        indexes = cur.fetchall()

print("\nDatabase indexes")

for table_name, index_name in indexes:
    print(f"{table_name}: {index_name}")

Indexes created successfully.

Database indexes
categories: categories_category_name_key
categories: categories_pkey
customers: customers_pkey
order_items: idx_order_items_order_id
order_items: idx_order_items_product_id
order_items: order_items_pkey
orders: idx_orders_customer_id
orders: idx_orders_order_date
orders: idx_orders_store_id
orders: orders_pkey
payments: payments_pkey
products: idx_products_category_id
products: products_pkey
promotions: promotions_pkey
returns: idx_returns_order_item_id
returns: returns_pkey
stores: stores_pkey


## 19. Create a read-only database role

The AI assistant should never connect to PostgreSQL using my database owner account.

I will create a separate analytics role that can read the tables but cannot change the data. This gives the project a database-level safety layer even if unsafe SQL is generated later.

In [22]:
from getpass import getpass
from psycopg import sql

role_name = "analytics_app"

# Enter a password for the read-only database role
role_password = getpass("Choose a password for analytics_app: ")

with psycopg.connect(dbname="ai_analytics") as conn:
    with conn.cursor() as cur:

        # Create the role if it does not already exist
        cur.execute("""
            SELECT 1
            FROM pg_roles
            WHERE rolname = %s;
        """, (role_name,))

        role_exists = cur.fetchone()

        if not role_exists:
            cur.execute(
                sql.SQL("CREATE ROLE {} LOGIN").format(
                    sql.Identifier(role_name)
                )
            )

        # Set or update the password
        cur.execute(
            sql.SQL("ALTER ROLE {} PASSWORD {}").format(
                sql.Identifier(role_name),
                sql.Literal(role_password)
            )
        )

        # Make sure this role does not have administrative privileges
        cur.execute(
            sql.SQL("""
                ALTER ROLE {}
                NOSUPERUSER
                NOCREATEDB
                NOCREATEROLE
                NOREPLICATION;
            """).format(
                sql.Identifier(role_name)
            )
        )

        # New sessions for this role start as read-only
        cur.execute(
            sql.SQL("""
                ALTER ROLE {}
                SET default_transaction_read_only = on;
            """).format(
                sql.Identifier(role_name)
            )
        )

        # Allow the role to connect and inspect the public schema
        cur.execute(
            sql.SQL("""
                GRANT CONNECT ON DATABASE ai_analytics TO {};
                GRANT USAGE ON SCHEMA public TO {};
            """).format(
                sql.Identifier(role_name),
                sql.Identifier(role_name)
            )
        )

        # Give read access to the current tables
        cur.execute(
            sql.SQL("""
                GRANT SELECT ON ALL TABLES IN SCHEMA public TO {};
            """).format(
                sql.Identifier(role_name)
            )
        )

        # Explicitly remove write permissions
        cur.execute(
            sql.SQL("""
                REVOKE INSERT, UPDATE, DELETE, TRUNCATE
                ON ALL TABLES IN SCHEMA public
                FROM {};
            """).format(
                sql.Identifier(role_name)
            )
        )

print("Read-only analytics role configured.")

Read-only analytics role configured.


In [23]:
# Test that the analytics role can read the database
with psycopg.connect(
    dbname="ai_analytics",
    user=role_name,
    password=role_password,
    host="localhost"
) as conn:

    with conn.cursor() as cur:
        cur.execute("SELECT COUNT(*) FROM orders;")
        order_count = cur.fetchone()[0]

        cur.execute("""
            SELECT
                current_user,
                current_setting('default_transaction_read_only');
        """)

        current_user, read_only_setting = cur.fetchone()


print("Connected as:", current_user)
print("Orders visible:", order_count)
print("Default transaction read-only:", read_only_setting)


# Now deliberately try a write operation
try:
    with psycopg.connect(
        dbname="ai_analytics",
        user=role_name,
        password=role_password,
        host="localhost"
    ) as conn:

        with conn.cursor() as cur:
            cur.execute("""
                UPDATE products
                SET list_price = list_price
                WHERE product_id = 1;
            """)

    print("Write test: FAILED - update was allowed")

except Exception as error:
    print("Write test: PASSED - update was blocked")
    print("Database response:", str(error).splitlines()[0])

Connected as: analytics_app
Orders visible: 25000
Default transaction read-only: on
Write test: PASSED - update was blocked
Database response: cannot execute UPDATE in a read-only transaction


## 20. Final database checks

Before finishing Day 1, I want to run one final check of the database setup.

This will confirm that the tables, data, relationships, indexes and read-only access are all in place before I start connecting the AI system.

In [24]:
final_checks = {}

with psycopg.connect(dbname="ai_analytics") as conn:
    with conn.cursor() as cur:

        # Check the number of tables
        cur.execute("""
            SELECT COUNT(*)
            FROM information_schema.tables
            WHERE table_schema = 'public'
              AND table_type = 'BASE TABLE';
        """)
        final_checks["9 tables exist"] = cur.fetchone()[0] == 9

        # Check the number of foreign-key relationships
        cur.execute("""
            SELECT COUNT(*)
            FROM information_schema.table_constraints
            WHERE constraint_schema = 'public'
              AND constraint_type = 'FOREIGN KEY';
        """)
        final_checks["8 foreign keys exist"] = cur.fetchone()[0] == 8

        # Check the main data volumes
        cur.execute("SELECT COUNT(*) FROM customers;")
        final_checks["5000 customers loaded"] = cur.fetchone()[0] == 5000

        cur.execute("SELECT COUNT(*) FROM orders;")
        final_checks["25000 orders loaded"] = cur.fetchone()[0] == 25000

        cur.execute("SELECT COUNT(*) FROM order_items;")
        final_checks["Order items loaded"] = cur.fetchone()[0] == 75104

        cur.execute("SELECT COUNT(*) FROM payments;")
        final_checks["25000 payments loaded"] = cur.fetchone()[0] == 25000

        cur.execute("SELECT COUNT(*) FROM returns;")
        final_checks["Returns loaded"] = cur.fetchone()[0] == 4832

        # Check the dataset date range
        cur.execute("""
            SELECT MIN(order_date), MAX(order_date)
            FROM orders;
        """)
        min_order_date, max_order_date = cur.fetchone()

        final_checks["Order dates inside dataset range"] = (
            min_order_date >= DATA_START_DATE
            and max_order_date <= DATA_END_DATE
        )

        # Check the indexes we deliberately added
        cur.execute("""
            SELECT COUNT(*)
            FROM pg_indexes
            WHERE schemaname = 'public'
              AND indexname LIKE 'idx_%';
        """)
        final_checks["7 analytics indexes exist"] = cur.fetchone()[0] == 7


# Check that the business glossary file exists
final_checks["Business glossary exists"] = glossary_path.exists()

# Check the controlled analysis date
final_checks["Analysis reference date set"] = (
    ANALYSIS_REFERENCE_DATE == date(2026, 8, 1)
)


print("Day 1 final checks\n")

for check_name, passed in final_checks.items():
    print(f"{check_name}: {passed}")

print()
print("All final checks passed:", all(final_checks.values()))

print()
print("Dataset order period:", min_order_date, "to", max_order_date)
print("Analysis reference date:", ANALYSIS_REFERENCE_DATE)

Day 1 final checks

9 tables exist: True
8 foreign keys exist: True
5000 customers loaded: True
25000 orders loaded: True
Order items loaded: True
25000 payments loaded: True
Returns loaded: True
Order dates inside dataset range: True
7 analytics indexes exist: True
Business glossary exists: True
Analysis reference date set: True

All final checks passed: True

Dataset order period: 2025-01-06 to 2026-07-31
Analysis reference date: 2026-08-01


In [25]:
settings_path = Path("../config/project_settings.json")

with open(settings_path, "r") as file:
    project_settings = json.load(file)

print("Random seed:", project_settings["random_seed"])
print("Customers:", project_settings["dataset_size"]["customers"])
print("Orders:", project_settings["dataset_size"]["orders"])
print(
    "Analysis reference date:",
    project_settings["date_range"]["analysis_reference_date"]
)

Random seed: 42
Customers: 5000
Orders: 25000
Analysis reference date: 2026-08-01
